In [ ]:
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
ARM, STEPS, EXTRA = "B", 1000, ""

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

man = [Path(p) for p in glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True)
       if (Path(p).parent / "code/defrost_graph/bilm/kaggle/train_sup.py").exists()]
assert man, "attach the private dataset kev-sup-v1"
D = man[0].parent
sh(f"{sys.executable} -m pip install -q 'transformers==5.17.0' 'peft==0.21.0' 'safetensors>=0.4.3'")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
OUT = Path(f"/kaggle/working/sup-{ARM}")
cgsa = f"--cgsa {D}/cgsa-adapter" if True else ""
src = "--sources " if "" else ""
t0 = time.time()
sh(f"cd {D}/code && PYTHONUNBUFFERED=1 torchrun --nproc_per_node 2 defrost_graph/bilm/kaggle/train_sup.py "
   f"--data {D}/data --mntp {D}/mntp-adapter {cgsa} {src} --steps {STEPS} --out {OUT} {EXTRA} 2>&1 | tee {OUT}.log")
print(f"done in {(time.time() - t0) / 60:.0f} min")
with zipfile.ZipFile(f"/kaggle/working/sup-{ARM}.zip", "w") as z:
    for p in OUT.rglob("*"):
        if p.is_file() and "last" not in p.parts:
            z.write(p, f"sup-{ARM}/{p.relative_to(OUT)}")
    z.write(f"{OUT}.log", f"sup-{ARM}/train.log")
